# Motive Raw CSV QC — Layers 1–5 + Layer 6 batch (v0.6)

Pipeline order: **L1 parse → L2 gaps → L4 artifact events → L3 window safety → L5 report**.

| Layer | Role |
|---|---|
| **L2** | Per-marker gaps and missingness (evidence) |
| **L4** | Kinematic artifact **events** on gap-safe segments (candidates only) |
| **L3** | Fixed windows — **safe for PCA/jPCA?** verdict using L2 gaps + L4 events |
| **L5** | `qc_report.md`, `qc_intervals`, `analysis_frame_mask` |

**Validation gates**
- **G0** — Layer 1–2 baseline
- **G1** — Artifact events plausible (L4) before window verdict (L3)
- **G2** — Window quality and gap overlap understood (L3)
- **G3** — Full report, intervals, and mask

Spectral screening is **disabled** in v0.5. Artifact defaults target **expressive movement** (looser velocity gate; accel/hold off). Outputs go to timestamped folders under `outputs/runs/`; includes `artifact_velocity_histogram.png`.

In [1]:
import sys
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import HTML, Image, Markdown, clear_output, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "motive_raw_qc.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import motive_raw_qc as mqc
from motive_qc.core import __version__
from motive_qc.output_tiers import resolve_run_output_dir

display(pd.DataFrame({"package": ["motive_qc", "motive_raw_qc"], "version": [__version__, __version__]}))

,package,version
0,motive_qc,0.6.0
1,motive_raw_qc,0.6.0


In [2]:
CONFIG_PATH = PROJECT_ROOT / "config.yaml"
config = mqc.load_config(CONFIG_PATH)
config["_base_dir"] = PROJECT_ROOT
config["reporting"]["stop_after_layer"] = 5
config["frame_quality"]["enabled"] = True
config["windows"]["enabled"] = True
config["artifacts"]["enabled"] = True
config.setdefault("spectral_screen", {})["enabled"] = False
config.setdefault("outputs", {})["tier"] = "essential"
config["outputs"]["write_frame_level_artifacts"] = False

layer1 = layer2 = layer3 = layer4 = layer5 = None
session = None
RUN_DIR = None
REPORT_PATH = None
BATCH_RESULT = None

art = config["artifacts"]
win = config["windows"]
spike = art.setdefault("single_frame_spike", {})
hold = art.setdefault("constant_position_hold", {})
methods = art.setdefault("methods", {})

vel_mad_w = widgets.FloatSlider(value=art.get("velocity_mad_multiplier", 11.0), min=3, max=15, step=0.5, description="Vel MAD:")
acc_mad_w = widgets.FloatSlider(value=art.get("acceleration_mad_multiplier", 11.0), min=3, max=15, step=0.5, description="Accel MAD:")
vel_pct_w = widgets.FloatSlider(value=art.get("velocity_percentile_threshold", 99.97), min=95, max=99.99, step=0.01, description="Vel pct:")
acc_pct_w = widgets.FloatSlider(value=art.get("acceleration_percentile_threshold", 99.97), min=95, max=99.99, step=0.01, description="Accel pct:")
spike_jump_w = widgets.FloatSlider(value=spike.get("min_jump_distance_m", 0.10), min=0.01, max=0.25, step=0.005, description="Spike jump m:")
hold_frames_w = widgets.IntSlider(value=hold.get("min_repeated_frames", 5), min=2, max=10, description="Hold min fr:")
accel_on_w = widgets.Checkbox(value=methods.get("acceleration_mad", False), description="Accel MAD on")
hold_on_w = widgets.Checkbox(value=methods.get("constant_position_hold", False), description="Hold detect on")
gap_flag_w = widgets.FloatSlider(value=win.get("flag_if_gap_at_least_seconds", 0.2), min=0.05, max=1.0, step=0.05, description="Win gap s:")
miss_pct_w = widgets.FloatSlider(value=win.get("flag_if_missing_marker_percent_above", 10.0), min=1, max=30, step=1, description="Win miss %:")

tuning_out = widgets.Output()
rerun_btn = widgets.Button(description="Re-run L4-L5", button_style="warning")


def _apply_tuning_to_config():
    art["velocity_mad_multiplier"] = vel_mad_w.value
    art["acceleration_mad_multiplier"] = acc_mad_w.value
    art["velocity_percentile_threshold"] = vel_pct_w.value
    art["acceleration_percentile_threshold"] = acc_pct_w.value
    spike["min_jump_distance_m"] = spike_jump_w.value
    hold["min_repeated_frames"] = hold_frames_w.value
    win["flag_if_gap_at_least_seconds"] = gap_flag_w.value
    win["flag_if_missing_marker_percent_above"] = miss_pct_w.value
    methods["acceleration_mad"] = accel_on_w.value
    methods["constant_position_hold"] = hold_on_w.value


def _run_l4_l5(_=None):
    global layer3, layer4, layer5, RUN_DIR, REPORT_PATH
    if session is None or layer2 is None:
        with tuning_out:
            clear_output(wait=True)
            print("Run G0 (L1-L2) first.")
        return
    _apply_tuning_to_config()
    config.pop("_run_output_dir", None)
    with tuning_out:
        clear_output(wait=True)
        layer4 = mqc.run_layer4_artifacts(session, layer2, config, verbose=True)
        layer3 = mqc.run_layer3_windows(session, layer2, layer4, config, verbose=True)
        layer5 = mqc.run_layer5_report(layer1, layer2, layer3, layer4, config, verbose=True)
        files = mqc.write_outputs(
            layer1, layer2, config,
            layer3_result=layer3,
            layer4_result=layer4,
            layer5_result=layer5,
            verbose=True,
        )
        RUN_DIR = Path(config["_run_output_dir"])
        REPORT_PATH = RUN_DIR / "qc_report.md"
        ev = layer4.tables.get("artifact_events", pd.DataFrame())
        art_s = layer4.tables.get("artifact_session_summary", pd.DataFrame())
        win_s = layer3.tables.get("window_quality_summary", pd.DataFrame())
        print(f"Wrote {len(files)} files to {RUN_DIR}")
        print(f"Artifact events: {len(ev)}")
        if not art_s.empty:
            r = art_s.iloc[0]
            print(f"Vel+accel overlap frames: {r.get('n_frames_both_velocity_and_acceleration', 0)}")
        if not win_s.empty:
            for _, row in win_s.iterrows():
                print(
                    f"{row['window_length_s']}s windows: {row['n_with_gap_overlap']} gap overlap, "
                    f"{row['n_with_artifact_events']} artifact overlap, "
                    f"{row['n_flagged_exclude']} exclude"
                )


rerun_btn.on_click(_run_l4_l5)
display(Markdown(
    "### L4-L5 tuning (Gaga defaults: looser velocity gate; accel/hold off)\n"
    "Higher **Vel MAD** / **Vel pct** -> fewer velocity flags. See artifact_velocity_histogram after run."
))
display(widgets.VBox([
    widgets.HBox([vel_mad_w, acc_mad_w]),
    widgets.HBox([vel_pct_w, acc_pct_w]),
    widgets.HBox([spike_jump_w, hold_frames_w]),
    widgets.HBox([accel_on_w, hold_on_w]),
    widgets.HBox([gap_flag_w, miss_pct_w]),
    rerun_btn,
    tuning_out,
]))

### L4-L5 tuning (Gaga defaults: looser velocity gate; accel/hold off)
Higher **Vel MAD** / **Vel pct** -> fewer velocity flags. See artifact_velocity_histogram after run.

In [3]:
from motive_qc.discovery import apply_session_to_config, discover_sessions

catalog = discover_sessions(config)
subject_opts = ["All"] + sorted(catalog["subject_id"].unique().tolist()) if not catalog.empty else ["All"]

subject_filter_dd = widgets.Dropdown(options=subject_opts, value=subject_opts[0], description="Subject:")
session_select = widgets.SelectMultiple(options=[], description="Sessions:", rows=8, layout=widgets.Layout(width="90%"))
refresh_btn = widgets.Button(description="Refresh catalog")
load_session_btn = widgets.Button(description="Load selected session", button_style="info")
batch_btn = widgets.Button(description="Run batch for PI", button_style="success")
session_out = widgets.Output()


def _filtered_catalog():
    if catalog.empty:
        return catalog
    if subject_filter_dd.value == "All":
        return catalog
    return catalog[catalog["subject_id"] == subject_filter_dd.value]


def _session_options():
    fc = _filtered_catalog()
    return [f"{r['session_id']} — {r['file_name']}" for _, r in fc.iterrows()]


def _row_from_label(label: str):
    sess_id = label.split(" — ", 1)[0]
    return _filtered_catalog().loc[_filtered_catalog()["session_id"] == sess_id].iloc[0]


def _update_session_list(*_):
    session_select.options = _session_options()


def _refresh_catalog(_=None):
    global catalog
    catalog = discover_sessions(config)
    opts = ["All"] + sorted(catalog["subject_id"].unique().tolist()) if not catalog.empty else ["All"]
    subject_filter_dd.options = opts
    _update_session_list()


def _load_selected_session(_=None):
    with session_out:
        clear_output(wait=True)
        if not session_select.value:
            print("Select a session first.")
            return
        label = list(session_select.value)[0]
        row = _row_from_label(label)
        global config
        config = apply_session_to_config(config, row)
        print(f"Loaded {row['subject_id']} / {row['session_id']}")
        print(f"CSV: {row['csv_path']}")


def _run_batch_for_pi(_=None):
    global BATCH_RESULT, layer1, layer2, layer3, layer4, layer5, session, RUN_DIR, REPORT_PATH
    with session_out:
        clear_output(wait=True)
        if not session_select.value:
            print("Select one or more sessions for batch.")
            return
        rows = [_row_from_label(lbl).to_dict() for lbl in session_select.value]
        print(f"Running Layer 6 batch on {len(rows)} session(s)...")
        BATCH_RESULT = mqc.run_batch(config, catalog_rows=rows, verbose=True)
        print(f"\nBatch folder: {BATCH_RESULT.batch_dir}")
        print(f"PI report: {BATCH_RESULT.report_paths.get('md')}")
        display(BATCH_RESULT.eda_df.head(20))


subject_filter_dd.observe(lambda _: _update_session_list(), names="value")
refresh_btn.on_click(_refresh_catalog)
load_session_btn.on_click(_load_selected_session)
batch_btn.on_click(_run_batch_for_pi)
_update_session_list()

display(Markdown("### Session picker & Layer 6 batch (PI deliverable: `dataset_eda_report.md`)"))
display(widgets.VBox([
    widgets.HBox([subject_filter_dd, refresh_btn]),
    session_select,
    widgets.HBox([load_session_btn, batch_btn]),
    session_out,
]))

### Session picker & Layer 6 batch (PI deliverable: `dataset_eda_report.md`)

## G0 — Layer 1–2 baseline

Parse CSV and compute gap/missingness evidence. Keeps `MotiveSession` in memory for L4 kinematics.

In [4]:
layer1 = mqc.run_layer1_parse(config, verbose=True)
session = layer1.session
layer2 = mqc.run_layer2_gaps(session, config, verbose=True)
summ = layer2.tables["session_summary"].iloc[0]
display(Markdown(f"**Preprocessing status:** `{summ['raw_qc_preprocessing_status']}` — {summ['raw_qc_status_reason']}"))

**Preprocessing status:** `poor` — Union labeled gap time (>= moderate) is 260.725s, at or above poor threshold (10.0s).

## Layer 4 — Artifact events (runs before L3)

### G1 checklist
- [ ] Events are clustered (duration, body segment, window linkage)
- [ ] No velocity computed across gap boundaries
- [ ] Wording remains *candidate* / *suggestive*, not confirmed filter detection

In [5]:
_apply_tuning_to_config()
layer4 = mqc.run_layer4_artifacts(session, layer2, config, verbose=True)
events = layer4.tables.get("artifact_events", pd.DataFrame())
art_summary = layer4.tables.get("artifact_session_summary", pd.DataFrame())
display(Markdown(f"**Artifact events:** {len(events)}"))
if not art_summary.empty:
    display(art_summary.T)
if not events.empty:
    display(events["method"].value_counts().to_frame("count"))
    display(events.head(10))

**Artifact events:** 216

,0
n_frame_candidates,216
n_events,216
n_single_frame_events,216
n_short_burst_events,0
n_sustained_events,0
n_frames_velocity_candidate,189
n_frames_acceleration_candidate,0
n_frames_both_velocity_and_acceleration,0
recommendation,Most detections are single-frame spikes; revie...


,count
method,
velocity_mad,216


,event_id,marker_name,body_region_group,method,start_frame,end_frame,start_time_s,end_time_s,duration_frames,duration_seconds,event_class,severity,near_gap,methods_in_event,peak_metric_value
0,E000001,T3_671:LHandOut,wrist_hand,velocity_mad,5375,5375,44.791667,44.791667,1,0.008333,single_frame,minor,False,velocity_mad,6.723767
1,E000002,T3_671:LHandOut,wrist_hand,velocity_mad,15784,15784,131.533333,131.533333,1,0.008333,single_frame,moderate,False,velocity_mad,8.533406
2,E000003,T3_671:LHandOut,wrist_hand,velocity_mad,29191,29191,243.258333,243.258333,1,0.008333,single_frame,minor,False,velocity_mad,6.501896
3,E000004,T3_671:LHandOut,wrist_hand,velocity_mad,29557,29557,246.308333,246.308333,1,0.008333,single_frame,minor,False,velocity_mad,6.543016
4,E000005,T3_671:LHandOut,wrist_hand,velocity_mad,30440,30440,253.666667,253.666667,1,0.008333,single_frame,minor,False,velocity_mad,6.677534
5,E000006,T3_671:LHandOut,wrist_hand,velocity_mad,30730,30730,256.083333,256.083333,1,0.008333,single_frame,minor,False,velocity_mad,6.697874
6,E000007,T3_671:LFArm,shoulder_upper_arm,velocity_mad,5401,5401,45.008333,45.008333,1,0.008333,single_frame,minor,False,velocity_mad,5.090017
7,E000008,T3_671:LFArm,shoulder_upper_arm,velocity_mad,15210,15210,126.750000,126.750000,1,0.008333,single_frame,minor,False,velocity_mad,4.735716
8,E000009,T3_671:LFArm,shoulder_upper_arm,velocity_mad,20517,20517,170.975000,170.975000,1,0.008333,single_frame,severe,False,velocity_mad,11.296193
9,E000010,T3_671:LFArm,shoulder_upper_arm,velocity_mad,20759,20759,172.991667,172.991667,1,0.008333,single_frame,moderate,False,velocity_mad,5.843828


## Layer 3 — Window safety (L2 gaps + L4 artifacts)

### G2 checklist
- [ ] Window durations match expected frame counts
- [ ] `n_gaps_overlapping` and `n_artifact_events_overlapping` populated
- [ ] `window_quality_label` reflects combined L2+L4 verdict
- [ ] No automatic exclusions applied

In [6]:
layer3 = mqc.run_layer3_windows(session, layer2, layer4, config, verbose=True)
win_summary = layer3.tables.get("window_quality_summary", pd.DataFrame())
display(win_summary)
w05 = layer3.tables.get("window_quality_0p5s", pd.DataFrame())
if not w05.empty:
  flagged = w05[w05["window_quality_label"] != "use"]
  display(Markdown(f"**Flagged 0.5s windows:** {len(flagged)} / {len(w05)}"))
  display(w05.head(10))

,window_length_s,table_name,n_windows,n_with_gap_overlap,n_with_artifact_events,n_flagged_caution,n_flagged_exclude,pct_session_in_flagged_windows
0,0.5,window_quality_0p5s,524,524,71,6,518,98.3340
1,1.0,window_quality_1p0s,262,262,50,0,262,99.1686


**Flagged 0.5s windows:** 524 / 524

,window_id,start_frame,end_frame,start_time_s,end_time_s,duration_frames,window_length_s,mean_missing_labeled_percent,max_missing_labeled_percent,n_gaps_overlapping,...,max_gap_duration_s,worst_gap_marker,critical_group_affected,n_artifact_events,n_sustained_artifact_events,max_artifact_event_duration_s,affected_body_groups,artifact_body_groups,window_quality_label,reason_codes
0,W00001,0,59,0.0,0.491667,60,0.5,3.703704,3.703704,2,...,18.85,T3_671:LThumb,True,0,0,0.0,fingers;thigh_knee,,exclude_or_review,LARGE_GAP_OVERLAP;CRITICAL_GROUP_GAP
1,W00002,60,119,0.5,0.991667,60,0.5,3.703704,3.703704,2,...,18.85,T3_671:LThumb,True,0,0,0.0,fingers;thigh_knee,,exclude_or_review,LARGE_GAP_OVERLAP;CRITICAL_GROUP_GAP
2,W00003,120,179,1.0,1.491667,60,0.5,3.703704,3.703704,2,...,18.85,T3_671:LThumb,True,0,0,0.0,fingers;thigh_knee,,exclude_or_review,LARGE_GAP_OVERLAP;CRITICAL_GROUP_GAP
3,W00004,180,239,1.5,1.991667,60,0.5,3.703704,3.703704,2,...,18.85,T3_671:LThumb,True,0,0,0.0,fingers;thigh_knee,,exclude_or_review,LARGE_GAP_OVERLAP;CRITICAL_GROUP_GAP
4,W00005,240,299,2.0,2.491667,60,0.5,3.703704,3.703704,2,...,18.85,T3_671:LThumb,True,0,0,0.0,fingers;thigh_knee,,exclude_or_review,LARGE_GAP_OVERLAP;CRITICAL_GROUP_GAP
5,W00006,300,359,2.5,2.991667,60,0.5,3.703704,3.703704,2,...,18.85,T3_671:LThumb,True,0,0,0.0,fingers;thigh_knee,,exclude_or_review,LARGE_GAP_OVERLAP;CRITICAL_GROUP_GAP
6,W00007,360,419,3.0,3.491667,60,0.5,3.703704,3.703704,2,...,18.85,T3_671:LThumb,True,0,0,0.0,fingers;thigh_knee,,exclude_or_review,LARGE_GAP_OVERLAP;CRITICAL_GROUP_GAP
7,W00008,420,479,3.5,3.991667,60,0.5,3.703704,3.703704,2,...,18.85,T3_671:LThumb,True,0,0,0.0,fingers;thigh_knee,,exclude_or_review,LARGE_GAP_OVERLAP;CRITICAL_GROUP_GAP
8,W00009,480,539,4.0,4.491667,60,0.5,3.703704,3.703704,2,...,18.85,T3_671:LThumb,True,0,0,0.0,fingers;thigh_knee,,exclude_or_review,LARGE_GAP_OVERLAP;CRITICAL_GROUP_GAP
9,W00010,540,599,4.5,4.991667,60,0.5,3.703704,3.703704,2,...,18.85,T3_671:LThumb,True,0,0,0.0,fingers;thigh_knee,,exclude_or_review,LARGE_GAP_OVERLAP;CRITICAL_GROUP_GAP


## Layer 5 — Report package

### G3 checklist
- [ ] `qc_report.md` fills all template sections
- [ ] `qc_intervals.csv` intervals have duration and `reason_human`
- [ ] `analysis_frame_mask` statuses align with gaps and window flags

In [7]:
layer5 = mqc.run_layer5_report(layer1, layer2, layer3, layer4, config, verbose=True)
config.pop("_run_output_dir", None)
files = mqc.write_outputs(
    layer1, layer2, config,
    layer3_result=layer3,
    layer4_result=layer4,
    layer5_result=layer5,
    verbose=True,
)
RUN_DIR = Path(config["_run_output_dir"])
REPORT_PATH = RUN_DIR / "qc_report.md"
print(f"Wrote {len(files)} files to {RUN_DIR}")

Wrote 27 files to C:\gaga_prj\Motive_QC\outputs\runs\T3_P1_R2_20260613_211114


In [ ]:
ALL_FIGURES = {}
for res in (layer2, layer3, layer4):
    if res:
        ALL_FIGURES.update(res.figures)

TABLE_OPTIONS = {
    "session_summary": lambda: layer2.tables["session_summary"],
    "gap_events": lambda: layer2.tables["gap_events"],
    "artifact_events": lambda: layer4.tables.get("artifact_events", pd.DataFrame()).head(500),
    "artifact_session_summary": lambda: layer4.tables.get("artifact_session_summary", pd.DataFrame()),
    "window_quality_summary": lambda: layer3.tables.get("window_quality_summary", pd.DataFrame()),
    "window_quality_0p5s": lambda: layer3.tables.get("window_quality_0p5s", pd.DataFrame()),
    "qc_intervals": lambda: layer5.tables.get("qc_intervals", pd.DataFrame()),
    "analysis_mask_summary": lambda: layer5.tables.get("analysis_mask_summary", pd.DataFrame()),
}

overview_out = widgets.Output()
tables_out = widgets.Output()
plots_out = widgets.Output()
report_out = widgets.Output()
messages_out = widgets.Output()

table_dd = widgets.Dropdown(options=list(TABLE_OPTIONS.keys()), description="Table:")
STATIC_PLOTS = sorted(k for k in ALL_FIGURES.keys() if not k.startswith("artifact_velocity_histogram"))
plot_dd = widgets.Dropdown(
    options=["artifact_velocity_histogram", *STATIC_PLOTS] if STATIC_PLOTS else ["artifact_velocity_histogram"],
    description="Plot:",
)
HIST_GROUPS = mqc.list_velocity_histogram_groups(session) if session is not None else ["all_labeled"]
segment_dd = widgets.Dropdown(options=HIST_GROUPS, description="Body segment:")


def render_overview():
    md = session.metadata
    summ = layer2.tables["session_summary"].iloc[0]
    art_s = layer4.tables.get("artifact_session_summary", pd.DataFrame())
    art_row = art_s.iloc[0].to_dict() if not art_s.empty else {}
    win_s = layer3.tables.get("window_quality_summary", pd.DataFrame())
    w05 = layer3.tables.get("window_quality_0p5s", pd.DataFrame())
    n_flagged = len(w05[w05["window_quality_label"] != "use"]) if not w05.empty else 0
    intervals = layer5.tables.get("qc_intervals", pd.DataFrame())
    mask_s = layer5.tables.get("analysis_mask_summary", pd.DataFrame())
    mask_lines = "<br>".join(f"{r['status']}: {r['n_frames']}" for _, r in mask_s.iterrows()) if not mask_s.empty else "(none)"
    win_lines = ""
    for _, row in win_s.iterrows():
        win_lines += (
            f"{row['window_length_s']}s: {row['n_windows']} windows, "
            f"{row.get('n_with_gap_overlap', 0)} gap overlap, "
            f"{row.get('n_with_artifact_events', 0)} artifact overlap, "
            f"{row.get('n_flagged_exclude', 0)} exclude<br>"
        )
    html = f"""
    <div style="font-family: sans-serif; line-height: 1.5;">
      <h3 style="margin-top:0;">Session</h3>
      <b>File:</b> {md['file_name']}<br>
      <b>Frames:</b> {md['total_frames_observed']} @ {md['effective_frame_rate_hz']} Hz<br>
      <b>Preprocessing QC:</b> {summ['raw_qc_preprocessing_status']}<br><br>
      <h3>Layer guide</h3>
      <b>L2</b> gaps/missingness | <b>L4</b> artifact events | <b>L3</b> window verdict (L2+L4) | <b>L5</b> mask/intervals<br><br>
      <h3>Artifact screening (L4)</h3>
      Events: {art_row.get('n_events', 0)} | single-frame: {art_row.get('n_single_frame_events', 0)} | sustained: {art_row.get('n_sustained_events', 0)}<br>
      Vel+accel overlap frames: {art_row.get('n_frames_both_velocity_and_acceleration', 0)}<br>
      {art_row.get('recommendation', '')}<br><br>
      <h3>Window safety (L3)</h3>
      {win_lines or '(no windows)'}
      Flagged 0.5s windows: {n_flagged} / {len(w05)}<br><br>
      <h3>Analysis mask (L5)</h3>
      {mask_lines}<br>
      QC intervals: {len(intervals)}<br><br>
      <b>Run folder:</b> {RUN_DIR}<br>
      <b>Report:</b> {REPORT_PATH}
    </div>
    """
    display(HTML(html))
    display(Markdown(f"**Status reason (L2):** {summ.get('raw_qc_status_reason', '')}"))


def render_table(name):
    display(TABLE_OPTIONS[name]())


def render_velocity_histogram(group: str):
    from motive_qc.plots import figure_velocity_artifact_histogram

    candidates = layer4.tables.get("artifact_candidates", pd.DataFrame()) if layer4 else pd.DataFrame()
    events = layer4.tables.get("artifact_events", pd.DataFrame()) if layer4 else pd.DataFrame()
    region = None if group == "all_labeled" else group
    dist = mqc.collect_session_velocity_distribution(session, config, region)
    flagged = mqc.flagged_velocity_speeds(candidates, session, region, artifact_events=events)
    fig = figure_velocity_artifact_histogram(dist, flagged, group)
    display(fig)
    plt.close(fig)


def render_plot(name):
    if name == "artifact_velocity_histogram":
        render_velocity_histogram(segment_dd.value)
        return
    if name in ALL_FIGURES:
        display(Image(filename=str(ALL_FIGURES[name])))


def render_report():
    if REPORT_PATH and REPORT_PATH.exists():
        display(Markdown(REPORT_PATH.read_text(encoding="utf-8")))


def render_messages():
    msgs = []
    for res in (layer1, layer2, layer3, layer4, layer5):
        if res:
            msgs.extend(res.messages)
    display(pd.DataFrame([{"severity": m.severity, "code": m.code, "message": m.message} for m in msgs]))


def _on_table(change):
    with tables_out:
        clear_output(wait=True)
        render_table(table_dd.value)


def _on_plot(change):
    with plots_out:
        clear_output(wait=True)
        render_plot(plot_dd.value)


def _on_segment(change):
    if plot_dd.value == "artifact_velocity_histogram":
        with plots_out:
            clear_output(wait=True)
            render_velocity_histogram(segment_dd.value)


def _segment_row_visible():
    segment_dd.layout.display = (
        None if plot_dd.value == "artifact_velocity_histogram" else "none"
    )


def _on_plot_segment_visibility(change):
    _segment_row_visible()
    _on_plot(change)


table_dd.observe(_on_table, names="value")
plot_dd.observe(_on_plot_segment_visibility, names="value")
segment_dd.observe(_on_segment, names="value")

with overview_out:
    render_overview()
with tables_out:
    render_table(table_dd.value)
plots_panel = widgets.VBox([widgets.HBox([plot_dd, segment_dd]), plots_out])
_segment_row_visible()
with plots_out:
    render_plot(plot_dd.value)
with report_out:
    render_report()
with messages_out:
    render_messages()

tabs = widgets.Tab(children=[overview_out, tables_out, plots_panel, report_out, messages_out])
for i, t in enumerate(["Overview", "Tables", "Plots", "Report", "Messages"]):
    tabs.set_title(i, t)
display(Markdown("### Results explorer (L1–5)"))
display(tabs)

### Results explorer (L1–5)

In [ ]:
validated_by_w = widgets.Text(description="Validated by:", layout=widgets.Layout(width="400px"))
decision_w = widgets.Dropdown(options=["pending", "approved", "needs correction"], description="Decision:")
l3_w = widgets.Dropdown(options=["pending", "approved"], description="Layer 3:")
l4_w = widgets.Dropdown(options=["pending", "approved"], description="Layer 4:")
l5_w = widgets.Dropdown(options=["pending", "approved"], description="Layer 5:")
notes_w = widgets.Textarea(description="Notes:", layout=widgets.Layout(width="90%", height="80px"))
log_btn = widgets.Button(description="Write validation log", button_style="primary")
signoff_out = widgets.Output()


def _write_log(_):
    with signoff_out:
        clear_output(wait=True)
        path = mqc.write_validation_log(
            layer1, layer2, config,
            log_path="docs/VALIDATION_LOG.md",
            decision=decision_w.value,
            validated_by=validated_by_w.value,
            notes=notes_w.value or "Notebook 02 v0.5 run.",
            layer3_decision=l3_w.value,
            layer4_decision=l4_w.value,
            layer5_decision=l5_w.value,
        )
        print(f"Wrote {path}")


log_btn.on_click(_write_log)
display(Markdown("### G3 — Validation sign-off"))
display(widgets.VBox([validated_by_w, decision_w, l3_w, l4_w, l5_w, notes_w, log_btn, signoff_out]))

## Configurable per-session QC plots

Gap **heatmap** (in-analysis labeled markers x time) and artifact **velocity histogram**, plus the focused deliverable tables (`gaps_over_0p5s`, `artifacts_by_segment`, `quarantined_markers`). Adjust the knobs at the top of the cell (downsampling, body-segment filter, histogram bins) and re-run.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from motive_qc.analysis_scope import analysis_labeled_marker_names

# --- knobs (edit and re-run) ---
HEATMAP_MAX_FRAMES = 1500      # downsample columns for display
HEATMAP_SEGMENT = "all"        # body_region_group (e.g. "wrist_hand") or "all"
HIST_SEGMENT = "all_labeled"   # velocity histogram group; see mqc.list_velocity_histogram_groups(session)
HIST_BINS = 80


def plot_gap_heatmap(session, config, max_frames=HEATMAP_MAX_FRAMES, segment="all"):
    inv = session.marker_inventory
    markers = analysis_labeled_marker_names(inv, config)
    if segment != "all":
        keep = set(inv.loc[inv["body_region_group"].astype(str) == segment, "marker_name"])
        markers = [m for m in markers if m in keep]
    if not markers:
        print("No in-analysis labeled markers for segment:", segment)
        return
    valid = session.valid_marker_frame.sel(marker=markers).values  # (frames, markers)
    missing = (~valid).T.astype(float)                              # markers x frames
    if missing.shape[1] > max_frames:
        step = int(np.ceil(missing.shape[1] / max_frames))
        missing = missing[:, ::step]
    fig, ax = plt.subplots(figsize=(12, max(3, len(markers) * 0.18)))
    ax.imshow(missing, aspect="auto", cmap="Reds", interpolation="nearest", vmin=0, vmax=1)
    ax.set_yticks(range(len(markers)))
    ax.set_yticklabels([m.split(":")[-1] for m in markers], fontsize=7)
    ax.set_xlabel(f"frame (downsampled to {missing.shape[1]} cols)")
    ax.set_title(f"Missing-frame heatmap (red = missing) - {segment}")
    plt.tight_layout()
    plt.show()


def plot_artifact_histogram(session, layer4, config, group=HIST_SEGMENT, bins=HIST_BINS):
    grp = None if group == "all_labeled" else group
    dist = mqc.collect_session_velocity_distribution(session, config, grp)
    speeds = np.asarray(dist.get("speeds_m_s", []), dtype=float)
    if speeds.size == 0:
        print("No gap-safe speeds for group:", group)
        return
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(speeds, bins=bins, color="#4a5568", alpha=0.85, label="all gap-safe speeds")
    ax.axvline(
        dist["vel_threshold_m_s"], color="#e53e3e", ls="--",
        label=f"threshold {dist['vel_threshold_m_s']:.2f} m/s",
    )
    ax.set_yscale("log")
    ax.set_xlabel("speed (m/s)")
    ax.set_ylabel("count (log)")
    ax.set_title(
        f"Artifact velocity histogram - {group} "
        f"(sigma={dist['vel_mad_multiplier']}, pct={dist['vel_percentile_config']})"
    )
    ax.legend()
    plt.tight_layout()
    plt.show()


plot_gap_heatmap(session, config, segment=HEATMAP_SEGMENT)
plot_artifact_histogram(session, layer4, config, group=HIST_SEGMENT)

display(Markdown("**Gaps >= 0.5 s by labeled marker**"))
display(layer5.tables["gaps_over_0p5s"])
display(Markdown("**Artifacts by body segment**"))
display(layer5.tables["artifacts_by_segment"])
display(Markdown("**Quarantined markers (excluded from missingness)**"))
display(layer2.tables.get("quarantined_markers", pd.DataFrame()))